# 🏦 Bank Marketing Deposit Prediction — MLOps Pipeline
**Nama:** Ivan Rahadian  
**Username Dicoding:** rahadianivan09

---

## Tentang Proyek

| | |
|---|---|
| **Dataset** | [Bank Marketing Dataset](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset) — UCI Bank Marketing, 11.162 baris, 16 fitur |
| **Masalah** | Bank ingin memprediksi apakah seorang nasabah akan berlangganan deposito berjangka (`deposit: yes/no`) berdasarkan data demografis dan riwayat kampanye marketing. |
| **Solusi** | Pipeline ML end-to-end menggunakan **TFX** dengan **Apache Beam** sebagai Pipeline Orchestrator. |
| **Metode Pengolahan** | One-hot encoding & vocabulary lookup (kategorik), z-score normalization (numerik) via komponen Transform. |
| **Arsitektur Model** | DNN klasifikasi biner, arsitektur ditentukan otomatis oleh **Keras Tuner**. Output sigmoid. |
| **Metrik Evaluasi** | BinaryAccuracy, AUC, Precision, Recall, ExampleCount via TFMA. |

## ⚙️ Cell 1 — Verifikasi Environment

Memastikan dependency utama tersedia dan versinya kompatibel sebelum menjalankan pipeline.

In [1]:
# ── CELL 1: Verifikasi Environment ───────────────
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import warnings
warnings.filterwarnings("ignore")

import tfx
import tensorflow as tf

print("TFX version  :", tfx.__version__)
print("TF version   :", tf.__version__)
print("PROTOCOL_BUFFERS:", os.environ.get("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"))

TFX version  : 1.14.0
TF version   : 2.13.1
PROTOCOL_BUFFERS: python


## 📦 Cell 2 — Import Library

Import seluruh library yang dibutuhkan untuk membangun pipeline TFX dengan Apache Beam sebagai orchestrator.

In [10]:
# ── CELL 2: Import Library ───────────────────────
import os
import pandas as pd
import tensorflow as tf
import tensorflow_model_analysis as tfma
import tfx
import glob

from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen,
    ExampleValidator, Transform, Trainer,
    Evaluator, Pusher, Tuner,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.orchestration import pipeline as tfx_pipeline
from tfx.orchestration import metadata
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

print("Semua library berhasil diimport!")

Semua library berhasil diimport!


## 🗂️ Cell 3 — Setup Path

Mendefinisikan seluruh path yang digunakan pipeline: data, modules, pipeline root, metadata, dan serving model.

In [3]:
# ── CELL 3: Setup Path ───────────────────────────
PROJECT_ROOT   = os.getcwd()
DATA_ROOT      = os.path.join(PROJECT_ROOT, "data")
TRANSFORM_MODULE = os.path.join(PROJECT_ROOT, "modules", "rahadianivan09_transform.py")
TRAINER_MODULE   = os.path.join(PROJECT_ROOT, "modules", "rahadianivan09_trainer.py")
PIPELINE_NAME  = "bank-deposit"
PIPELINE_ROOT  = os.path.join(PROJECT_ROOT, "rahadianivan09-pipeline")
METADATA_PATH  = os.path.join(PIPELINE_ROOT, "metadata", "metadata.db")
SERVING_MODEL  = os.path.join(PIPELINE_ROOT, "serving_model")

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(PIPELINE_ROOT, exist_ok=True)
os.makedirs(os.path.dirname(METADATA_PATH), exist_ok=True)
os.makedirs(SERVING_MODEL, exist_ok=True)

print(f"Project root   : {PROJECT_ROOT}")
print(f"Data           : {DATA_ROOT}")
print(f"Pipeline root  : {PIPELINE_ROOT}")
print(f"Metadata       : {METADATA_PATH}")
print(f"Serving model  : {SERVING_MODEL}")

Project root   : /workspaces/rahadianivan09-mlpipeline2
Data           : /workspaces/rahadianivan09-mlpipeline2/data
Pipeline root  : /workspaces/rahadianivan09-mlpipeline2/rahadianivan09-pipeline
Metadata       : /workspaces/rahadianivan09-mlpipeline2/rahadianivan09-pipeline/metadata/metadata.db
Serving model  : /workspaces/rahadianivan09-mlpipeline2/rahadianivan09-pipeline/serving_model


## 📊 Cell 4 — Eksplorasi Dataset

Melihat distribusi data sebelum masuk ke pipeline untuk memahami karakteristik dataset.

In [4]:
# ── CELL 4: Eksplorasi Dataset ───────────────────
df = pd.read_csv(os.path.join(DATA_ROOT, "bank.csv"))

# Konversi label deposit ke int jika masih string
if df['deposit'].dtype == object:
    df['deposit'] = (df['deposit'] == 'yes').astype(int)
    df.to_csv(os.path.join(DATA_ROOT, "bank.csv"), index=False)
    print("Label 'deposit' dikonversi ke int (yes=1, no=0)")

print(f"Total data      : {len(df)}")
print(f"Fitur           : {list(df.columns)}")
print(f"Deposit (1=yes) : {df['deposit'].sum()} ({df['deposit'].mean()*100:.1f}%)")
print(f"No deposit (0)  : {len(df) - df['deposit'].sum()}")
df.head()

Total data      : 11162
Fitur           : ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit']
Deposit (1=yes) : 5289 (47.4%)
No deposit (0)  : 5873


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,1
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,1
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,1
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,1
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,1


## 🔧 Cell 5 — Definisi Pipeline (`create_pipeline`)

Fungsi `create_pipeline()` membungkus seluruh komponen TFX (ExampleGen hingga Pusher) menjadi satu objek `Pipeline` yang siap dijalankan oleh Apache Beam.

Komponen yang digunakan:
| Komponen | Fungsi |
|---|---|
| ExampleGen | Membaca `bank.csv`, split train 80% / eval 20% |
| StatisticsGen | Statistik deskriptif seluruh fitur |
| SchemaGen | Infer schema otomatis sebagai kontrak validasi |
| ExampleValidator | Deteksi anomali terhadap schema |
| Transform | One-hot encoding & z-score normalization |
| Tuner | Hyperparameter search otomatis (Keras Tuner) |
| Trainer | Training DNN dengan best hyperparameter |
| Resolver | Ambil model blessed terbaru sebagai baseline |
| Evaluator | Evaluasi TFMA, bless jika lolos threshold |
| Pusher | Deploy model ke `serving_model/` jika BLESSED |

In [5]:
# ── CELL 5: Definisi create_pipeline() ───────────
def create_pipeline(
    pipeline_name: str,
    pipeline_root: str,
    data_root: str,
    transform_module: str,
    trainer_module: str,
    serving_model_dir: str,
    metadata_path: str,
) -> tfx_pipeline.Pipeline:
    """Membuat objek Pipeline TFX dengan seluruh komponen.

    Args:
        pipeline_name: nama pipeline.
        pipeline_root: direktori root penyimpanan artifact pipeline.
        data_root: direktori dataset CSV.
        transform_module: path modul transform.
        trainer_module: path modul trainer.
        serving_model_dir: direktori output serving model.
        metadata_path: path file SQLite metadata.

    Returns:
        tfx_pipeline.Pipeline siap dijalankan BeamDagRunner.
    """

    # ── ExampleGen ────────────────────────────────
    example_gen = CsvExampleGen(input_base=data_root)

    # ── StatisticsGen ─────────────────────────────
    statistics_gen = StatisticsGen(
        examples=example_gen.outputs["examples"]
    )

    # ── SchemaGen ─────────────────────────────────
    schema_gen = SchemaGen(
        statistics=statistics_gen.outputs["statistics"]
    )

    # ── ExampleValidator ──────────────────────────
    example_validator = ExampleValidator(
        statistics=statistics_gen.outputs["statistics"],
        schema=schema_gen.outputs["schema"],
    )

    # ── Transform ─────────────────────────────────
    transform = Transform(
        examples=example_gen.outputs["examples"],
        schema=schema_gen.outputs["schema"],
        module_file=os.path.abspath(transform_module),
    )

    # ── Tuner ─────────────────────────────────────
    tuner = Tuner(
        module_file=os.path.abspath(trainer_module),
        examples=transform.outputs["transformed_examples"],
        transform_graph=transform.outputs["transform_graph"],
        schema=schema_gen.outputs["schema"],
        train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=200),
        eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=100),
    )

    # ── Trainer ───────────────────────────────────
    trainer = Trainer(
        module_file=os.path.abspath(trainer_module),
        examples=transform.outputs["transformed_examples"],
        transform_graph=transform.outputs["transform_graph"],
        schema=schema_gen.outputs["schema"],
        hyperparameters=tuner.outputs["best_hyperparameters"],
        train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=100),
        eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=50),
    )

    # ── Resolver ──────────────────────────────────
    model_resolver = Resolver(
        strategy_class=LatestBlessedModelStrategy,
        model=Channel(type=Model),
        model_blessing=Channel(type=ModelBlessing),
    )

    # ── Evaluator ─────────────────────────────────
    eval_config = tfma.EvalConfig(
        model_specs=[
            tfma.ModelSpec(
                label_key="deposit",
                signature_name="serving_default",
            )
        ],
        slicing_specs=[tfma.SlicingSpec()],
        metrics_specs=[
            tfma.MetricsSpec(
                metrics=[
                    tfma.MetricConfig(
                        class_name="BinaryAccuracy",
                        threshold=tfma.MetricThreshold(
                            value_threshold=tfma.GenericValueThreshold(
                                lower_bound={"value": 0.50}
                            )
                        ),
                    ),
                    tfma.MetricConfig(
                        class_name="AUC",
                        threshold=tfma.MetricThreshold(
                            value_threshold=tfma.GenericValueThreshold(
                                lower_bound={"value": 0.70}
                            )
                        ),
                    ),
                    tfma.MetricConfig(class_name="Precision"),
                    tfma.MetricConfig(class_name="Recall"),
                    tfma.MetricConfig(class_name="ExampleCount"),
                ]
            )
        ],
    )

    evaluator = Evaluator(
        examples=example_gen.outputs["examples"],
        model=trainer.outputs["model"],
        baseline_model=model_resolver.outputs["model"],
        eval_config=eval_config,
    )

    # ── Pusher ────────────────────────────────────
    pusher = Pusher(
        model=trainer.outputs["model"],
        model_blessing=evaluator.outputs["blessing"],
        push_destination=pusher_pb2.PushDestination(
            filesystem=pusher_pb2.PushDestination.Filesystem(
                base_directory=serving_model_dir
            )
        ),
    )

    components = [
        example_gen,
        statistics_gen,
        schema_gen,
        example_validator,
        transform,
        tuner,
        trainer,
        model_resolver,
        evaluator,
        pusher,
    ]

    return tfx_pipeline.Pipeline(
        pipeline_name=pipeline_name,
        pipeline_root=pipeline_root,
        components=components,
        metadata_connection_config=metadata.sqlite_metadata_connection_config(
            metadata_path
        ),
    )

print("create_pipeline() berhasil didefinisikan!")


create_pipeline() berhasil didefinisikan!


## 🚀 Cell 6 — Jalankan Pipeline dengan Apache Beam

Menjalankan seluruh pipeline menggunakan `BeamDagRunner` sebagai Pipeline Orchestrator.  
`BeamDagRunner` akan mengorkestrasi eksekusi setiap komponen secara berurutan menggunakan **Apache Beam DirectRunner**.

In [6]:
# ── CELL 6: Run Pipeline dengan BeamDagRunner ────
BeamDagRunner().run(
    create_pipeline(
        pipeline_name=PIPELINE_NAME,
        pipeline_root=PIPELINE_ROOT,
        data_root=DATA_ROOT,
        transform_module=TRANSFORM_MODULE,
        trainer_module=TRAINER_MODULE,
        serving_model_dir=SERVING_MODEL,
        metadata_path=METADATA_PATH,
    )
)
print("\n✅ Pipeline selesai dijalankan via Apache Beam!")

Trial 3 Complete [00h 00m 07s]
val_auc: 0.7676944732666016

Best val_auc So Far: 0.7676944732666016
Total elapsed time: 00h 00m 25s
INFO:tensorflow:Oracle triggered exit


INFO:tensorflow:Oracle triggered exit


Results summary
Results in /workspaces/rahadianivan09-mlpipeline2/rahadianivan09-pipeline/Tuner/.system/executor_execution/7/.temp/7/bank_tuning
Showing 10 best trials
Objective(name="val_auc", direction="max")

Trial 2 summary
Hyperparameters:
units_1: 128
units_2: 32
units_3: 16
dropout: 0.2
learning_rate: 0.01
Score: 0.7676944732666016

Trial 0 summary
Hyperparameters:
units_1: 128
units_2: 128
units_3: 64
dropout: 0.2
learning_rate: 0.001
Score: 0.7514151930809021

Trial 1 summary
Hyperparameters:
units_1: 128
units_2: 128
units_3: 48
dropout: 0.30000000000000004
learning_rate: 0.001
Score: 0.747431218624115


Processing ./rahadianivan09-pipeline/_wheels/tfx_user_code_Trainer-0.0+9c973d072e5bc4a52d15df6fcbfc76f3994823fec83308480ecc0824d8f88eb1-py3-none-any.whl



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


...

INFO:tensorflow:Assets written to: /workspaces/rahadianivan09-mlpipeline2/rahadianivan09-pipeline/Trainer/model/8/Format-Serving/assets


INFO:tensorflow:Assets written to: /workspaces/rahadianivan09-mlpipeline2/rahadianivan09-pipeline/Trainer/model/8/Format-Serving/assets


Model berhasil disimpan ke: /workspaces/rahadianivan09-mlpipeline2/rahadianivan09-pipeline/Trainer/model/8/Format-Serving
Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`



✅ Pipeline selesai dijalankan via Apache Beam!


## ✅ Cell 7 — Verifikasi Hasil Pipeline

Mengecek struktur folder `rahadianivan09-pipeline/` untuk memastikan seluruh artifact pipeline dan serving model berhasil dibuat

In [7]:
# ── CELL 7: Verifikasi Struktur Pipeline ─────────
print(f"Struktur folder pipeline ({PIPELINE_ROOT}):\n")
for root, dirs, files in os.walk(PIPELINE_ROOT):
    # Skip folder artifact yang terlalu dalam
    level = root.replace(PIPELINE_ROOT, "").count(os.sep)
    if level > 3:
        continue
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level <= 2:
        subindent = "  " * (level + 1)
        for file in files:
            print(f"{subindent}{file}")

Struktur folder pipeline (/workspaces/rahadianivan09-mlpipeline2/rahadianivan09-pipeline):

rahadianivan09-pipeline/
  SchemaGen/
    .system/
      executor_execution/
    schema/
      4/
  metadata/
    metadata.db
  _wheels/
    tfx_user_code_Transform-0.0+9c973d072e5bc4a52d15df6fcbfc76f3994823fec83308480ecc0824d8f88eb1-py3-none-any.whl
    tfx_user_code_Trainer-0.0+9c973d072e5bc4a52d15df6fcbfc76f3994823fec83308480ecc0824d8f88eb1-py3-none-any.whl
    tfx_user_code_Tuner-0.0+9c973d072e5bc4a52d15df6fcbfc76f3994823fec83308480ecc0824d8f88eb1-py3-none-any.whl
  Tuner/
    .system/
      executor_execution/
    tuner_results/
      7/
    best_hyperparameters/
      7/
  Transform/
    post_transform_schema/
      5/
    transform_graph/
      5/
    updated_analyzer_cache/
      5/
    pre_transform_schema/
      5/
    pre_transform_stats/
      5/
    .system/
      executor_execution/
    post_transform_anomalies/
      5/
    transformed_examples/
      5/
    post_transform_stats/


## ✅ Cell 8 — Verifikasi Serving Model
Cell terakhir memverifikasi bahwa model berhasil di-push dengan menampilkan struktur direktori serving_model/.
Struktur yang diharapkan:
serving_model/
└── <timestamp>/
    ├── saved_model.pb
    ├── fingerprint.pb
    └── variables/
        ├── variables.index
        └── variables.data-00000-of-00001
Adanya direktori bertimestamp mengonfirmasi model telah berhasil di-export dalam format SavedModel dan siap di-serve via TensorFlow Serving atau Docker.

In [8]:
# ── CELL 8: Verifikasi Serving Model ─────────────
print(f"Struktur serving_model/:\n")
model_found = False
for root, dirs, files in os.walk(SERVING_MODEL):
    level = root.replace(SERVING_MODEL, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")
        model_found = True

if model_found:
    print("\n✅ Serving model berhasil dibuat oleh Pusher!")
else:
    print("\n⚠️  Serving model belum ada — cek apakah model lolos evaluasi (BLESSED).")

Struktur serving_model/:

serving_model/
  1782455796/
    saved_model.pb
    fingerprint.pb
    keras_metadata.pb
    variables/
      variables.index
      variables.data-00000-of-00001
    assets/
      job_vocab
      default_vocab
      marital_vocab
      education_vocab
      contact_vocab
      housing_vocab
      loan_vocab
      month_vocab

✅ Serving model berhasil dibuat oleh Pusher!


## ✅ Cell 9 - Verifikasi Hasil Evaluasi Model
Mengecek status BLESSED dari Evaluator dan menampilkan metrik evaluasi (Accuracy, AUC, Precision, Recall).

In [11]:
# ── CELL 9: Verifikasi Hasil Evaluasi Model ─────────
# Cari folder blessing & evaluation terbaru
blessing_dir = sorted(glob.glob(os.path.join(PIPELINE_ROOT, "Evaluator", "blessing", "*")))[-1]
eval_dir = sorted(glob.glob(os.path.join(PIPELINE_ROOT, "Evaluator", "evaluation", "*")))[-1]

is_blessed = tf.io.gfile.exists(os.path.join(blessing_dir, "BLESSED"))
print("Model Status:", "✅ BLESSED" if is_blessed else "❌ NOT BLESSED")

eval_result = tfma.load_eval_result(eval_dir)
print(eval_result.slicing_metrics)

Model Status: ✅ BLESSED
[((), {'': {'': {'accuracy': {'doubleValue': 0.6945043206214905}, 'auc': {'doubleValue': 0.7534463472405184}, 'precision': {'doubleValue': 0.6873846153846154}, 'recall': {'doubleValue': 0.6408491107286288}, 'loss': {'doubleValue': 0.5929156541824341}, 'example_count': {'doubleValue': 3712.0}, 'binary_accuracy': {'doubleValue': 0.6945043103448276}}}})]
